# 🧮 The Deutsch–Jozsa Algorithm
### A First Glimpse of Quantum Advantage

---

## 📖 Background & Motivation

The **Deutsch–Jozsa algorithm** (1992, extended by Deutsch & Jozsa) was one of the first quantum algorithms to demonstrate a provable exponential speedup over any classical deterministic algorithm.

### The Problem
Imagine a black box (called an **oracle**) that computes some function:

$$f: \{0,1\}^n \rightarrow \{0, 1\}$$

We are **promised** that $f$ is one of two types:
- **Constant**: $f(x) = 0$ for all inputs, or $f(x) = 1$ for all inputs
- **Balanced**: exactly half the inputs give 0, the other half give 1

**Goal**: Determine which type it is — using as few queries to the oracle as possible.

### Classical vs Quantum
| Approach | Queries Needed |
|---|---|
| Classical (deterministic) | Up to $2^{n-1}+1$ queries |
| Classical (randomistic) | A few queries (but with error probability) |
| **Quantum (Deutsch–Jozsa)** | **Exactly 1 query** |

This is an **exponential speedup** — a landmark result in quantum computing!

---

## 🧱 Key Quantum Concepts You Need

### 1. Qubits
Unlike classical bits (0 or 1), a qubit can be in a **superposition**:
$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle \quad \text{where } |\alpha|^2 + |\beta|^2 = 1$$

### 2. The Hadamard Gate (H)
The Hadamard gate creates superposition:
$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}} = |+\rangle$$
$$H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}} = |-\rangle$$

Applied to $n$ qubits all in $|0\rangle$:
$$H^{\otimes n}|0\rangle^{\otimes n} = \frac{1}{\sqrt{2^n}} \sum_{x \in \{0,1\}^n} |x\rangle$$
This puts the system in an **equal superposition of all possible inputs** at once — quantum parallelism!

### 3. Phase Kickback
The oracle encodes $f(x)$ using a trick called phase kickback. For an ancilla qubit in state $|-\rangle$:
$$U_f|x\rangle|-\rangle = (-1)^{f(x)}|x\rangle|-\rangle$$
The result of $f(x)$ gets encoded as a **phase** on the input qubit, not changing the ancilla at all.

---

## 🔬 The Algorithm — Step by Step

For $n = 2$ (2 input qubits + 1 ancilla), the circuit looks like:

```
|0⟩ ──H──────[  Oracle  ]──H── Measure
|0⟩ ──H──────[  U_f     ]──H── Measure
|1⟩ ──H──────[          ]──── (discard)
              (ancilla)
```

**Step 1 — Initialization:**
Start with $|0\rangle^{\otimes n}|1\rangle$ (input qubits in $|0\rangle$, ancilla in $|1\rangle$)

**Step 2 — Apply H to all qubits:**
$$\frac{1}{\sqrt{2^n}} \sum_x |x\rangle \cdot |-\rangle$$

**Step 3 — Apply the oracle $U_f$:**
$$\frac{1}{\sqrt{2^n}} \sum_x (-1)^{f(x)}|x\rangle \cdot |-\rangle$$

**Step 4 — Apply H to input qubits again:**
Due to quantum interference:
- If $f$ is **constant** → all input qubits measure as $|0\rangle$
- If $f$ is **balanced** → at least one input qubit measures as $|1\rangle$

**Step 5 — Measure:** A single measurement reveals the answer!

---

## 💡 Why Does It Work? Interference!

After the second Hadamard, the amplitude for measuring all zeros is:
$$\text{amplitude}(|0\rangle^n) = \frac{1}{2^n} \sum_{x} (-1)^{f(x)}$$

- If $f$ is **constant**: all $(-1)^{f(x)}$ have the same sign → they add up → amplitude = ±1
- If $f$ is **balanced**: half are +1, half are -1 → they cancel → amplitude = 0

This is **quantum interference** doing the work!


In [2]:
# ============================================================
#  CELL 1: Imports & Setup
# ============================================================
# We use:
#   cirq   — Google's quantum computing framework
#   numpy  — numerical arrays and matrix operations
!pip install cirq
import cirq
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print("✅ Libraries loaded successfully!")
print(f"   cirq version : {cirq.__version__}")
print(f"   numpy version: {np.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 19.1 MB/s eta 0:00:00
✅ Libraries loaded successfully!
   cirq version : 1.6.1
   numpy version: 2.0.2


## 🔲 Understanding the Oracle

The oracle $U_f$ is represented as a **unitary matrix** acting on all $n+1$ qubits (inputs + ancilla).

For $n = 2$, we have $2^3 = 8$ basis states: $|000\rangle, |001\rangle, \ldots, |111\rangle$, so the matrix is **8×8**.

### Constant Oracles
- **Constant-0**: $f(x) = 0$ for all $x$ → the oracle acts as the identity on input qubits and flips ancilla if $f(x)=1$ (but it doesn't here). In phase kickback terms, no phase is added.
- **Constant-1**: $f(x) = 1$ for all $x$ → every input state picks up a $(-1)$ phase.

### Balanced Oracles
- $f(x) = 0$ for half the inputs, $f(x) = 1$ for the other half.
- A common example: $f(x_0, x_1) = x_0$ (depends only on the first qubit — balanced since half of all inputs have $x_0=0$, half have $x_0=1$).

### Your Input Matrix
When you enter a matrix below, make sure it is:
1. **8×8** (64 values)
2. **Unitary**: $U^\dagger U = I$ (columns and rows are orthonormal)

Use the helper `verify_unitary(matrix)` provided below to check!

In [3]:
# ============================================================
#  CELL 2: Helper Functions
# ============================================================

def verify_unitary(matrix, tol=1e-6):
    """
    Check whether a matrix is unitary: U†U = I.
    A valid quantum gate MUST be unitary.
    """
    product = matrix.conj().T @ matrix
    identity = np.eye(matrix.shape[0])
    is_unitary = np.allclose(product, identity, atol=tol)
    if is_unitary:
        print("✅ Matrix is unitary — valid quantum gate!")
    else:
        print("❌ Matrix is NOT unitary — invalid quantum gate!")
        print("   Max deviation from identity:", np.max(np.abs(product - identity)))
    return is_unitary


def create_oracle_gate(oracle_matrix):
    """Wrap a numpy matrix as a cirq gate."""
    return cirq.MatrixGate(oracle_matrix)


def input_unitary_matrix():
    """
    Prompt the user for 64 comma-separated values,
    reshape into an 8x8 numpy array, and validate.
    """
    print("Enter 64 comma-separated values for the 8x8 unitary matrix:")
    print("(Tip: try the example matrices provided in the markdown cell below)")
    raw = input().strip().split(',')
    if len(raw) != 64:
        raise ValueError(f"Expected 64 values, got {len(raw)}")
    matrix = np.array([float(v) for v in raw]).reshape(8, 8)
    verify_unitary(matrix)
    return matrix


def determine_function_type(measurement_outcomes_list):
    """
    Determine if the oracle encodes a constant or balanced function.

    Rule:
      - If ALL measurements of the input qubits are [0, 0]  → constant
      - Otherwise (at least one qubit measured as 1)        → balanced
    """
    # Majority vote across all runs for robustness
    all_zero = all(
        all(bit == 0 for bit in outcome)
        for outcome in measurement_outcomes_list
    )
    return "constant" if all_zero else "balanced"


print("✅ Helper functions defined.")

✅ Helper functions defined.


## 📋 Example Oracle Matrices

Copy and paste any of the following when prompted.

---

### Example 1 — Balanced Oracle (f depends on first qubit)
 It swaps |0⟩↔|1⟩ on the first qubit, encoding f(x₀,x₁) = x₀.

```
0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
```
Expected output: **balanced**

---

### Example 2 — Constant Oracle (f = 0, identity matrix)
f(x) = 0 for every input → the oracle does nothing.

```
1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
```
Expected output: **constant**

---

### Example 3 — Balanced Oracle (f depends on second qubit)
f(x₀,x₁) = x₁

```
0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
```
Expected output: **balanced**

In [4]:
# ============================================================
#  CELL 3: Build the Deutsch–Jozsa Circuit
# ============================================================
#
#  Circuit structure (n=2 input qubits + 1 ancilla):
#
#  qubit[0] : |0⟩ ──H──[Oracle]──H──M
#  qubit[1] : |0⟩ ──H──[Oracle]──H──M
#  qubit[2] : |0⟩──X──H──[Oracle]────── (ancilla, not measured)
#
#  The X gate on qubit[2] flips it to |1⟩ before H,
#  putting it in the |-⟩ state needed for phase kickback.

def deutsch_jozsa_circuit(oracle_gate):
    """
    Construct the Deutsch–Jozsa circuit for n=2 input qubits.

    Parameters
    ----------
    oracle_gate : cirq.Gate
        A 3-qubit gate (8×8 unitary) representing the oracle U_f.

    Returns
    -------
    cirq.Circuit
    """
    qubits = cirq.LineQubit.range(3)  # qubits[0], qubits[1] = input; qubits[2] = ancilla
    circuit = cirq.Circuit()

    # Step 1: Flip ancilla |0⟩ → |1⟩
    circuit.append(cirq.X(qubits[2]))

    # Step 2: Hadamard on ALL qubits
    #   Input qubits: |0⟩ → |+⟩  (superposition of 0 and 1)
    #   Ancilla:      |1⟩ → |-⟩  (needed for phase kickback)
    circuit.append([cirq.H(q) for q in qubits])

    # Step 3: Apply oracle (one single query!)
    circuit.append(oracle_gate(*qubits))

    # Step 4: Hadamard on input qubits only (NOT ancilla)
    #   This converts phase differences back into measurable amplitudes
    circuit.append([cirq.H(q) for q in qubits[:2]])

    # Step 5: Measure input qubits
    circuit.append([cirq.measure(q, key=f'result_{i}') for i, q in enumerate(qubits[:2])])

    return circuit


print("✅ Circuit builder defined.")

✅ Circuit builder defined.


In [5]:
# ============================================================
#  CELL 4: Run the Algorithm
# ============================================================
#
#  We run the circuit multiple times (shots) to demonstrate
#  that the result is DETERMINISTIC for Deutsch–Jozsa:
#  every shot should give the same measurement outcome.

NUM_SHOTS = 10  # Number of times we run the circuit

# --- Input the oracle matrix ---
oracle_matrix = input_unitary_matrix()

# --- Create the oracle gate ---
oracle_gate = create_oracle_gate(oracle_matrix)

# --- Build the circuit ---
circuit = deutsch_jozsa_circuit(oracle_gate)

# --- Simulate ---
simulator = cirq.Simulator()
measurement_outcomes_list = []

for shot in range(NUM_SHOTS):
    result = simulator.run(circuit)
    outcome = [int(result.measurements[f'result_{i}'][0][0]) for i in range(2)]
    measurement_outcomes_list.append(outcome)

# --- Classify ---
function_type = determine_function_type(measurement_outcomes_list)

# --- Display Results ---
print("\n" + "="*55)
print("  DEUTSCH–JOZSA RESULT")
print("="*55)
print(f"  Shots run       : {NUM_SHOTS}")
print(f"  All outcomes    : {measurement_outcomes_list}")
print(f"  Unique outcomes : {list(set(map(tuple, measurement_outcomes_list)))}")
print(f"  ⟹  Function is  : {function_type.upper()}")
print("="*55)
print()
if function_type == "constant":
    print("  Interpretation: All input qubits measured |0⟩.")
    print("  The oracle applies the SAME output for every input.")
else:
    print("  Interpretation: At least one input qubit measured |1⟩.")
    print("  The oracle outputs 0 for exactly half and 1 for the other half.")

# --- Print Circuit Diagram ---
print("\n" + "-"*55)
print("  CIRCUIT DIAGRAM")
print("-"*55)
print(circuit)

Enter 64 comma-separated values for the 8x8 unitary matrix:
(Tip: try the example matrices provided in the markdown cell below)
1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
✅ Matrix is unitary — valid quantum gate!

  DEUTSCH–JOZSA RESULT
  Shots run       : 10
  All outcomes    : [[0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, 0]]
  Unique outcomes : [(0, 0)]
  ⟹  Function is  : CONSTANT

  Interpretation: All input qubits measured |0⟩.
  The oracle applies the SAME output for every input.

-------------------------------------------------------
  CIRCUIT DIAGRAM
-------------------------------------------------------
              ┌                       ┐
              │1. 0. 0. 0. 0. 0. 0. 0.│
              │0. 1. 0. 0. 0. 0. 0. 0.│
              │0. 0. 1. 0. 0. 0. 0. 0.│
0: ───H───────│0. 0. 0. 1. 0. 0. 0. 0.│───H───M('result_0')───
              │0. 0. 0. 0. 1. 0. 0. 0.

In [6]:
# ============================================================
#  CELL 5: Statevector Walkthrough (Educational)
# ============================================================
#
#  This cell simulates the circuit WITHOUT measurements so
#  we can inspect the EXACT quantum state at each step.
#  This is a powerful tool for understanding the algorithm.

def print_statevector(label, state, n_qubits=3, threshold=1e-6):
    """Pretty-print a statevector, omitting near-zero amplitudes."""
    print(f"\n{'─'*50}")
    print(f"  {label}")
    print(f"{'─'*50}")
    for i, amp in enumerate(state):
        if abs(amp) > threshold:
            bits = format(i, f'0{n_qubits}b')
            print(f"  |{bits}⟩  →  {amp:.4f}")


# Build the circuit without final measurements (so we can inspect state)
qubits = cirq.LineQubit.range(3)
circuit_no_measure = cirq.Circuit()

# Step 1: X on ancilla
circuit_no_measure.append(cirq.X(qubits[2]))
state1 = simulator.simulate(circuit_no_measure).final_state_vector
print_statevector("After X on ancilla  |001⟩", state1)

# Step 2: H on all
circuit_no_measure.append([cirq.H(q) for q in qubits])
state2 = simulator.simulate(circuit_no_measure).final_state_vector
print_statevector("After H on all qubits (superposition)", state2)

# Step 3: Oracle
circuit_no_measure.append(oracle_gate(*qubits))
state3 = simulator.simulate(circuit_no_measure).final_state_vector
print_statevector("After Oracle U_f (phases encoded)", state3)

# Step 4: H on input qubits
circuit_no_measure.append([cirq.H(q) for q in qubits[:2]])
state4 = simulator.simulate(circuit_no_measure).final_state_vector
print_statevector("After final H on input qubits (ready to measure)", state4)

print("\n" + "─"*50)
print("  OBSERVATION:")
print("  - For a CONSTANT oracle: only |00x⟩ states have non-zero amplitude.")
print("  - For a BALANCED oracle: |00x⟩ has zero amplitude; other states dominate.")


──────────────────────────────────────────────────
  After X on ancilla  |001⟩
──────────────────────────────────────────────────
  |001⟩  →  1.0000+0.0000j

──────────────────────────────────────────────────
  After H on all qubits (superposition)
──────────────────────────────────────────────────
  |000⟩  →  0.3536+0.0000j
  |001⟩  →  -0.3536+0.0000j
  |010⟩  →  0.3536+0.0000j
  |011⟩  →  -0.3536+0.0000j
  |100⟩  →  0.3536+0.0000j
  |101⟩  →  -0.3536+0.0000j
  |110⟩  →  0.3536+0.0000j
  |111⟩  →  -0.3536+0.0000j

──────────────────────────────────────────────────
  After Oracle U_f (phases encoded)
──────────────────────────────────────────────────
  |000⟩  →  0.3536+0.0000j
  |001⟩  →  -0.3536+0.0000j
  |010⟩  →  0.3536+0.0000j
  |011⟩  →  -0.3536+0.0000j
  |100⟩  →  0.3536+0.0000j
  |101⟩  →  -0.3536+0.0000j
  |110⟩  →  0.3536+0.0000j
  |111⟩  →  -0.3536+0.0000j

──────────────────────────────────────────────────
  After final H on input qubits (ready to measure)
─────────────────

## 📊 Classical vs Quantum: How Big Is the Advantage?

For a function with $n$ input bits:

| $n$ | Classical queries needed | Quantum queries needed |
|-----|--------------------------|------------------------|
| 1   | 2                        | 1                      |
| 2   | 3                        | 1                      |
| 10  | 513                      | 1                      |
| 100 | $2^{99}+1 \approx 6.3 \times 10^{29}$ | 1       |
| $n$ | $2^{n-1}+1$              | **1**                  |

The quantum algorithm solves the problem in **exactly one oracle query**, always.

---

## ⚠️ Important Caveats for Students

1. **The problem is artificial.** The promise that $f$ is either constant or balanced is a strong constraint. Relaxing it removes the clean speedup.

2. **Classical probabilistic algorithms are fast too.** With a small number of random queries (say, 3–5), a classical algorithm can determine the function type with high probability. Deutsch–Jozsa beats *deterministic* classical algorithms.

3. **It's still historically important.** This algorithm paved the way for Shor's (factoring) and Grover's (search) algorithms, which provide speedups on practically relevant problems.

---

## 🧪 Exercises for Students

1. **Modify NUM_SHOTS** in Cell 4 to 1. Is the algorithm still correct with a single shot? Why?

2. **Inspect the statevector** in Cell 5 for both a constant and balanced oracle. Can you identify the interference pattern?


3. **Extend to $n=3$**: What changes in the circuit? (Hint: you need a 4-qubit system — 3 input + 1 ancilla — and a 16×16 oracle matrix.)

4. **Noise**: What happens if you add a small rotation error after the first Hadamard? Use `cirq.Rz(rads=0.1)` and observe the effect on the result.


## ✅ Summary

| Concept | Description |
|---|---|
| **Superposition** | H gate spreads a qubit over all basis states simultaneously |
| **Phase kickback** | The oracle writes f(x) as a phase, not a bit flip |
| **Interference** | The second H converts phases into measurable probabilities |
| **Single query** | Quantum parallelism evaluates f on all inputs at once |
| **Measurement** | All-zeros → constant; any-one → balanced |

Deutsch–Jozsa is often described as a "proof of concept" for quantum computing — it may not solve a real-world problem, but it rigorously shows that quantum computers can be *exponentially faster* than classical ones for certain tasks.

---
*Built with [Cirq](https://quantumai.google/cirq) — Google's open-source quantum computing framework.*